This notebook provides a workflow for defining a regular grid over administrative boundaries. 
The process includes loading administrative territory data, calculating grid extents, generating a 50 km grid, and exporting the result for further analysis.

In [ ]:
import geopandas as gpd
from shapely.geometry import box

In [ ]:
# set URL for administrative boundaries GeoJSON, load the data into a GeoDataFrame
admin_url = 'https://data.gov.lv/dati/dataset/7bb04db9-97ce-4a30-b93a-10ba8dafd104/resource/3ded58bd-c0dc-419a-97ff-59ba45a7b1b0/download/administrativas_teritorijas_2021.geojson'
gdf = gpd.read_file(admin_url)

In [ ]:
# calculate the bounding box for the grid, ensure it aligns with a 50 km grid, export the result
minx, miny, maxx, maxy = gdf.total_bounds
minx = minx - (minx % 50000)
miny = miny - (miny % 50000)
maxx = maxx + (50000 - (maxx % 50000))
maxy = maxy + (50000 - (maxy % 50000))

grid = gpd.GeoDataFrame({
    'geometry': [box(x, y, x + 50000, y + 50000) for x in range(int(minx), int(maxx), 50000)
                 for y in range(int(miny), int(maxy), 50000)]
}, crs=3059)
grid = grid[grid.intersects(gdf.union_all())]
grid['grid_id'] = range(len(grid))

grid.to_file('../data/grid_50km_epsg3059.geojson', driver='GeoJSON')